In [1]:
# === Notebook専用: サンプル割当の ARI/NMI/AMI を run-root 全体から一括算出 ===
from pathlib import Path
import pandas as pd, numpy as np, re

# ★環境に合わせて設定
BASE     = "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No"
RUN_ROOT = "20251026_under_25clusters"   # ここを親にして再帰探索
OUT_NAME = "STEP3.3_eval_labels_20251027"  # 出力フォルダ名（run-root直下）

try:
    from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, adjusted_mutual_info_score
    SKLEARN_OK=True
except Exception:
    SKLEARN_OK=False

ID_CANDIDATES  = {"id","sample","sample_id","sno","sid","index","name","SampleID","ID"}
LABEL_PRIOR    = ["cluster","label","assign","assignment","cluster_label","clusterid","cluster_id","k_label","k"]

def is_sample_assignment(df):
    for c in df.columns:
        if c in ID_CANDIDATES or str(c).lower() in {x.lower() for x in ID_CANDIDATES}:
            u=df[c].nunique(dropna=True)
            if len(df)==0: return False
            if u/max(1,len(df))>=0.3: return True
    return False

def detect_id_col(df):
    for c in df.columns:
        if str(c).lower() in {x.lower() for x in ID_CANDIDATES}: return c
    return df.columns[0]

def detect_label_col(df,id_col):
    for p in LABEL_PRIOR:
        for c in df.columns:
            if str(c).lower()==p: return c
    not_id=[c for c in df.columns if c!=id_col]
    n=len(df); best=None; bests=-1
    for c in not_id:
        u=df[c].nunique(dropna=True)
        is_intlike = pd.api.types.is_integer_dtype(df[c]) or pd.api.types.is_categorical_dtype(df[c])
        s=(1 if is_intlike else 0)+(1 if 2<=u<=max(2,n//2) else 0)
        if s>bests: best,bests=c,s
    return best

def parse_mode_index(fname):
    m=re.search(r"ClusterAssign_([^_]+)_([^_]+)_(OH|FP)_", fname)
    if m: return m.group(1),m.group(2),m.group(3)
    m2=re.search(r"ClusterAssign_(.*)_(OH|FP)_", fname)
    if m2:
        toks=m2.group(1).split("_")
        if len(toks)>=2: return toks[-2],toks[-1],m2.group(2)
    return "","","OH" if "_OH_" in fname else "FP"

def compute_pair(df_oh, df_fp):
    id_oh=detect_id_col(df_oh); id_fp=detect_id_col(df_fp)
    lab_oh=detect_label_col(df_oh,id_oh); lab_fp=detect_label_col(df_fp,id_fp)
    ids=sorted(set(df_oh[id_oh]).intersection(set(df_fp[id_fp])))
    if not ids: return None
    y_oh=df_oh.set_index(id_oh).loc[ids][lab_oh].astype(str).values
    y_fp=df_fp.set_index(id_fp).loc[ids][lab_fp].astype(str).values
    kA=int(pd.Series(y_oh).nunique()); kB=int(pd.Series(y_fp).nunique())
    if kA<2 or kB<2: return None
    if SKLEARN_OK:
        ARI=float(adjusted_rand_score(y_oh,y_fp))
        NMI=float(normalized_mutual_info_score(y_oh,y_fp))
        AMI=float(adjusted_mutual_info_score(y_oh,y_fp))
    else:
        # 簡易フォールバック
        def _ari(y_true,y_pred):
            y_true=np.asarray(y_true); y_pred=np.asarray(y_pred)
            _,lt=np.unique(y_true,return_inverse=True); _,lp=np.unique(y_pred,return_inverse=True)
            cont=pd.crosstab(pd.Series(lt),pd.Series(lp)).values
            ai=cont.sum(1); bj=cont.sum(0); n=cont.sum()
            comb=lambda x:(x*(x-1))//2
            sum_c=comb(cont).sum(); sum_a=comb(ai).sum(); sum_b=comb(bj).sum(); total=comb(n)
            exp=(sum_a*sum_b)/total if total>0 else 0.0
            maxv=0.5*(sum_a+sum_b); denom=maxv-exp
            return float((sum_c-exp)/denom) if denom!=0 else 0.0
        def _ent(p): p=np.asarray(p,float); p=p[p>0]; return -float(np.sum(p*np.log(p)))
        def _nmi(y_true,y_pred):
            y_true=np.asarray(y_true); y_pred=np.asarray(y_pred)
            _,lt=np.unique(y_true,return_inverse=True); _,lp=np.unique(y_pred,return_inverse=True)
            n=len(lt); cont=pd.crosstab(pd.Series(lt),pd.Series(lp)).values.astype(float)
            pi=cont.sum(1)/n; pj=cont.sum(0)/n; pij=cont/n
            with np.errstate(divide='ignore',invalid='ignore'):
                logt=np.log(pij/(pi[:,None]*pj[None,:])); logt[~np.isfinite(logt)]=0.0
                mi=float(np.nansum(pij*logt))
            h1=_ent(pi); h2=_ent(pj); denom=(h1+h2)/2.0 if h1+h2>0 else 1.0
            return float(mi/denom) if denom>0 else 0.0
        ARI=_ari(y_oh,y_fp); NMI=_nmi(y_oh,y_fp); AMI=NMI
    return dict(n_overlap=len(ids),kA_OH=kA,kB_FP=kB,ARI=ARI,NMI=NMI,AMI=AMI)

# 探索
base=Path(BASE).resolve(); run_root=(base/RUN_ROOT).resolve()
out_root = (run_root/OUT_NAME).resolve(); (out_root/"reports").mkdir(parents=True,exist_ok=True)

files_oh = sorted(run_root.glob("**/OH/04_assignments/ClusterAssign_*_OH_*.csv"))
files_fp = sorted(run_root.glob("**/FP/04_assignments/ClusterAssign_*_FP_*.csv"))

# sample割当だけ拾う
files_oh_s=[]; files_fp_s=[]
for p in files_oh:
    try:
        if is_sample_assignment(pd.read_csv(p,nrows=1000)): files_oh_s.append(p)
    except: pass
for p in files_fp:
    try:
        if is_sample_assignment(pd.read_csv(p,nrows=1000)): files_fp_s.append(p)
    except: pass

# mode/index でペアリング
def latest_by_mode_index(paths):
    d={}
    for p in sorted(paths):
        mode,index,side = parse_mode_index(p.name)
        d.setdefault((mode,index,side),p)
    return d
oh_map=latest_by_mode_index(files_oh_s)
fp_map=latest_by_mode_index(files_fp_s)

results=[]
for (mode,index,side), ohp in oh_map.items():
    if side!="OH": continue
    fpp = fp_map.get((mode,index,"FP"))
    if fpp is None: continue
    try:
        df_oh=pd.read_csv(ohp); df_fp=pd.read_csv(fpp)
        met=compute_pair(df_oh,df_fp)
        if met: results.append({"mode":mode,"index":index,"OH_file":str(ohp),"FP_file":str(fpp),**met})
    except Exception as e:
        results.append({"mode":mode,"index":index,"OH_file":str(ohp),"FP_file":str(fpp),"error":str(e)})

if results:
    df=pd.DataFrame(results).sort_values(["mode","index"])
    df.to_csv(out_root/"reports/ARI_NMI_AMI_summary_ASIS_sample_pairs.csv",index=False,encoding="utf-8-sig")
    print("Saved:", out_root/"reports/ARI_NMI_AMI_summary_ASIS_sample_pairs.csv")
else:
    print("No sample-assignment pairs detected anywhere under run-root.")

print("\n=== Done ===")
print("Reports folder:", out_root/"reports")

Saved: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters/STEP3.3_eval_labels_20251027/reports/ARI_NMI_AMI_summary_ASIS_sample_pairs.csv

=== Done ===
Reports folder: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters/STEP3.3_eval_labels_20251027/reports
